# Kimi K2.6 Cookbook

This notebook is a practical starting point for building with Kimi K2.6 through Moonshot AI's OpenAI-compatible API.

What is covered:
- setup and authentication
- basic chat completions
- streaming responses
- framework integration (LangChain)
- tool-enabled agent loop
- long-context patterns
- multimodal image and video request shapes
- K2.6-specific parameter and tool-calling constraints

Source: https://platform.kimi.ai/docs/guide/kimi-k2-6-quickstart/

## 0) Setup and Installation

In [ ]:
# Run this once in a fresh environment.
# %pip install -U "openai>=1.0" langchain langchain-openai

In [ ]:
import os
from openai import OpenAI

MODEL = "kimi-k2.6"
BASE_URL = os.getenv("MOONSHOT_BASE_URL", "https://api.moonshot.ai/v1")
API_KEY = os.getenv("MOONSHOT_API_KEY")

if not API_KEY:
    raise ValueError("Set MOONSHOT_API_KEY before running this notebook.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f"Using model: {MODEL}")
print(f"Base URL: {BASE_URL}")

## 1) Basic Usage with Provider SDK

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise software engineering assistant."},
        {"role": "user", "content": "Give 5 checks before shipping a backend API to production."},
    ],
)
print(response.choices[0].message.content)

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain idempotency keys with one concrete example."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

### Thinking mode toggle

K2.6 supports a `thinking` request field. By default it is enabled.

In [ ]:
response_no_thinking = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a short release note title."}],
    thinking={"type": "disabled"},
)
print(response_no_thinking.choices[0].message.content)

## 2) Framework Integration

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
)

msg = llm.invoke("Draft a rollout checklist for enabling a new LLM in CI pipelines.")
print(msg.content)

## 3) Building Agents

In [ ]:
import json

def get_deploy_risk(service: str) -> dict:
    lookup = {"billing-api": "medium", "auth-api": "high", "cache-api": "low"}
    return {"service": service, "risk": lookup.get(service, "unknown")}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_deploy_risk",
            "description": "Return deployment risk level for a service.",
            "parameters": {
                "type": "object",
                "properties": {"service": {"type": "string"}},
                "required": ["service"],
            },
        },
    }
]

In [ ]:
messages = [
    {"role": "system", "content": "You are a release assistant. Use tools before deciding."},
    {"role": "user", "content": "Should we deploy auth-api this week?"},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)
assistant_message = first.choices[0].message
messages.append(assistant_message.model_dump())

if assistant_message.tool_calls:
    for tool_call in assistant_message.tool_calls:
        args = json.loads(tool_call.function.arguments)
        result = get_deploy_risk(service=args["service"])
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            }
        )

second = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice="auto")
print(second.choices[0].message.content)

## 4) Advanced Applications

In [ ]:
# Long-context pattern: summarize chunks first, then synthesize final answer.

def chunk_text(text: str, size: int = 3000):
    return [text[i:i + size] for i in range(0, len(text), size)]

document = """
Paste a long design doc, incident report, or code review notes here.
"""

partials = []
for idx, chunk in enumerate(chunk_text(document), start=1):
    partial = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Summarize key technical points and open risks."},
            {"role": "user", "content": f"Chunk {idx}:\n{chunk}"},
        ],
    )
    partials.append(partial.choices[0].message.content)

final = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Merge these summaries into one actionable plan."},
        {"role": "user", "content": "\n\n".join(partials)},
    ],
)
print(final.choices[0].message.content)

## 5) K2.6 Multimodal Request Shapes

In [ ]:
import base64

# Image example. Replace the local file path before running.
# image_path = "kimi.png"
# with open(image_path, "rb") as f:
#     image_data = f.read()
# image_ext = image_path.split(".")[-1]
# image_url = f"data:image/{image_ext};base64,{base64.b64encode(image_data).decode('utf-8')}"
#
# completion = client.chat.completions.create(
#     model=MODEL,
#     messages=[
#         {"role": "system", "content": "You are Kimi."},
#         {
#             "role": "user",
#             "content": [
#                 {"type": "image_url", "image_url": {"url": image_url}},
#                 {"type": "text", "text": "Describe the image in technical detail."},
#             ],
#         },
#     ],
# )
# print(completion.choices[0].message.content)

In [ ]:
# Video example. Replace the local file path before running.
# video_path = "kimi.mp4"
# with open(video_path, "rb") as f:
#     video_data = f.read()
# video_ext = video_path.split(".")[-1]
# video_url = f"data:video/{video_ext};base64,{base64.b64encode(video_data).decode('utf-8')}"
#
# completion = client.chat.completions.create(
#     model=MODEL,
#     messages=[
#         {"role": "system", "content": "You are Kimi."},
#         {
#             "role": "user",
#             "content": [
#                 {"type": "video_url", "video_url": {"url": video_url}},
#                 {"type": "text", "text": "Describe what happens in this video."},
#             ],
#         },
#     ],
# )
# print(completion.choices[0].message.content)

## 6) K2.6 Parameter and Tool Compatibility Notes

- Keep `tool_choice` as `auto` or `none` when thinking is enabled.
- For multi-step tool calling with thinking enabled, preserve assistant reasoning fields in context.
- K2.6 uses fixed sampling values for several parameters. Avoid setting custom values for temperature, top_p, n, presence_penalty, and frequency_penalty unless docs explicitly allow it.
- Supported image formats: png, jpeg, webp, gif.
- Supported video formats: mp4, mpeg, mov, avi, x-flv, mpg, webm, wmv, 3gpp.
- Recommended max resolution: image up to 4k (4096x2160), video up to 2k (2048x1080).
- For large media, prefer file upload instead of base64 because of request body size limits.

---

You now have a complete baseline for Kimi K2.6: chat, streaming, tools, framework integration, long-context handling, and multimodal request patterns.